# Mission Tasking Service — Quickstart

Run each cell top-to-bottom. Cells that start with `!` are shell commands.

**Prerequisites before starting:**
- Docker Desktop is running
- You have your `ANTHROPIC_API_KEY` handy
- `uv` is installed (`~/.local/bin/uv`)

## 1. Set your API key

Fill in your key below. This writes `.env` from the example template.

In [ ]:
import os
import shutil

ANTHROPIC_API_KEY = "sk-ant-..."   # <-- paste your key here
LANGSMITH_API_KEY = ""             # optional — leave empty to skip LangSmith tracing

shutil.copy(".env.example", ".env")

with open(".env", "r") as f:
    env = f.read()

env = env.replace("ANTHROPIC_API_KEY=", f"ANTHROPIC_API_KEY={ANTHROPIC_API_KEY}")
env = env.replace("LANGSMITH_API_KEY=", f"LANGSMITH_API_KEY={LANGSMITH_API_KEY}")

with open(".env", "w") as f:
    f.write(env)

print(".env written ✓")

## 2. Install Python dependencies

In [ ]:
!~/.local/bin/uv sync --extra dev

## 3. Run the test suite (offline — no API key needed)

The safety core (physics model + kernel validator) and graph wiring tests all run without a live database or LLM.

In [ ]:
!~/.local/bin/uv run pytest -v

## 4. Start the full stack with Docker Compose

This brings up: MTS service, Postgres/PostGIS, Redis, OTel collector, Prometheus, Grafana.

> **Docker Desktop must be running before this cell.**

In [ ]:
!docker compose -f deploy/docker/docker-compose.yml up --build -d

## 5. Wait for services to be healthy

In [ ]:
import time, urllib.request, urllib.error

url = "http://localhost:8000/healthz"
for i in range(30):
    try:
        urllib.request.urlopen(url, timeout=2)
        print(f"MTS is up ✓  (attempt {i+1})")
        break
    except Exception:
        print(f"waiting... ({i+1}/30)")
        time.sleep(3)
else:
    print("MTS did not come up in time — check: docker compose logs mts")

## 6. Seed the database (operating areas + drone profiles)

In [ ]:
!docker compose -f deploy/docker/docker-compose.yml exec mts python scripts/seed_db.py

## 7. Compile a mission plan

Send a natural-language command and get back a structured `MissionPlan`.

In [ ]:
import json, urllib.request

payload = {
    "command": "Patrol the yard perimeter at 60 meters with EO sensor and return to base.",
    "area_id": "yard-simple",
    "operator_clearance": "STANDARD",
    "drone_state": {
        "drone_profile_id": "long-endurance-quad",
        "battery_pct": 100.0
    }
}

req = urllib.request.Request(
    "http://localhost:8000/v1/missions:compile",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read())

print(json.dumps(result, indent=2))

In [ ]:
# Pull out the key fields
plan = result["plan"]
print(f"Status        : {plan['status']}")
print(f"Mission ID    : {plan['mission_id']}")
print(f"Legs          : {len(plan['legs'])}")
print(f"Duration      : {plan['total_duration_s']:.0f}s ({plan['total_duration_s']/60:.1f} min)")
print(f"Battery used  : {plan['total_battery_pct']:.1f}%")
print(f"Reserve       : {plan['battery_reserve_pct']:.1f}%")
print(f"Repair loops  : {result['repair_loops']}")
print(f"\nReasoning:\n{plan['reasoning_trace']}")

## 8. Try a mission that should be rejected (NFZ violation)

In [ ]:
payload_bad = {
    "command": "Fly directly over the grain silo.",
    "area_id": "farmland-complex",
    "operator_clearance": "STANDARD",
    "drone_state": {
        "drone_profile_id": "long-endurance-quad",
        "battery_pct": 100.0
    }
}

req2 = urllib.request.Request(
    "http://localhost:8000/v1/missions:compile",
    data=json.dumps(payload_bad).encode(),
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req2, timeout=120) as resp:
    result_bad = json.loads(resp.read())

plan_bad = result_bad["plan"]
print(f"Status  : {plan_bad['status']}")
print(f"Reasons : {plan_bad['rejection_reasons']}")

## 9. Approve a ready plan

In [ ]:
mission_id = plan["mission_id"]  # from cell 7

if plan["status"] == "READY_FOR_APPROVAL":
    approval = {
        "mission_id": mission_id,
        "approve": True,
        "operator_note": "Looks good, proceed."
    }
    req3 = urllib.request.Request(
        "http://localhost:8000/v1/missions:approve",
        data=json.dumps(approval).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req3, timeout=30) as resp:
        print(json.dumps(json.loads(resp.read()), indent=2))
else:
    print(f"Plan status is '{plan['status']}' — nothing to approve.")

## 10. Verify the approved plan (execution simulation)

In [ ]:
req4 = urllib.request.Request(
    f"http://localhost:8000/v1/missions/{mission_id}:verify",
    method="POST",
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(req4, timeout=30) as resp:
    verify = json.loads(resp.read())

print(json.dumps(verify, indent=2))

## 11. Check Prometheus metrics

In [ ]:
with urllib.request.urlopen("http://localhost:8000/metrics") as resp:
    metrics = resp.read().decode()

# Print only MTS-specific lines
for line in metrics.splitlines():
    if line.startswith("mts_") and not line.startswith("#"):
        print(line)

## 12. Open dashboards

- **Grafana**: http://localhost:3000 — login `admin` / `admin`, import `deploy/grafana-dashboard.json`
- **Prometheus**: http://localhost:9090
- **MTS API docs**: http://localhost:8000/docs

## 13. Tear down

In [ ]:
!docker compose -f deploy/docker/docker-compose.yml down -v